# Python scripts

Work through this notebook from top to bottom, **on the course server**, with
a terminal open next to it. Some cells here create files; the terminal must
see the same files, so the notebook and the terminal must be on the same
computer.

So far you have run Python in notebook cells. A **script** is Python code in
a plain text file ending in `.py`. You run the whole file with one command in
the terminal. Scripts are how Python runs on servers and clusters, how you
run one analysis on many files, and how you share an analysis that others can
run with one command.

## Your first script

In the terminal, go to your `week-05` directory (make it if you skipped
lesson 2) and create a script with nano:

```
$ cd ~/week-05
$ nano hello.py
```

Type this line, save with Ctrl-O and exit with Ctrl-X:

```python
print("Hello from a script")
```

Run it:

```
$ python3 hello.py
Hello from a script
```

The command is `python3`, and its argument is the name of the script. (On
the course server, the command is `python3`, not `python`. On other computers
it may be `python`.)

## A script only shows what you `print`

In a notebook, the value of the last line of a cell is shown automatically.
In a script, nothing is shown unless you `print` it. A script containing just

```python
2 + 3
```

computes 5 and then prints nothing at all. Write `print(2 + 3)` instead.

## Writing a script from a notebook: `%%writefile`

Typing long scripts in nano is slow. A notebook cell that starts with the
line `%%writefile name.py` does not run its code. Instead, it saves the rest
of the cell into the file `name.py`, in the same directory as the notebook.

Run this cell:

In [ ]:
%%writefile hello2.py
name = "Anna"
print(f"Hello, {name}")

Now, in the terminal, go to the directory where this notebook is, and run
the script:

```
$ python3 hello2.py
Hello, Anna
```

Change the name in the cell, run the cell again, and run the script again.

## Writing a file from Python

Python can write files, too. This is how an analysis saves its results.

In [ ]:
with open("greeting.txt", "w") as f:
    f.write("Hello, file!\n")

- `open("greeting.txt", "w")` opens the file for **w**riting. If it exists, it
  is replaced (like `>` in the terminal).
- `with ... as f:` gives the open file the name `f`, and closes it properly
  at the end of the block.
- `f.write(...)` writes a string. It does not add a line break by itself:
  `"\n"` is the *newline* character, which ends a line.

Check in the terminal with `cat greeting.txt`.

Where did the file go? Into the **working directory** of the notebook, which
is the directory the notebook is in:

In [ ]:
import os

print(os.getcwd())

## Reading a file in Python

Opening without `"w"` opens the file for reading. `.read()` gives the whole
file as one string:

In [ ]:
with open("greeting.txt") as f:
    text = f.read()

print(repr(text))

`repr` shows the string with its quotes, so you can see the `\n` at the
end. `.strip()` (from week 4) removes it:

In [ ]:
print(repr(text.strip()))

A file can also be read line by line with a `for` loop. Here we first write
three sequences, one per line, and then read them back:

In [ ]:
with open("sequences.txt", "w") as f:
    f.write("ATGCGC\n")
    f.write("GGCATTA\n")
    f.write("TTAG\n")

with open("sequences.txt") as f:
    for line in f:
        seq = line.strip()
        print(seq, len(seq))

## Command-line arguments: `sys.argv`

When you run `python3 hello.py`, the words after `python3` are **arguments**.
A script can read them from the list `sys.argv`:

In [ ]:
%%writefile args.py
import sys

print(sys.argv)

Run it in the terminal with some arguments:

```
$ python3 args.py one 2 three.txt
['args.py', 'one', '2', 'three.txt']
```

- `sys.argv[0]` is the name of the script itself.
- `sys.argv[1]` is the first argument, `sys.argv[2]` the second, and so on.
- All arguments are **strings**, even `2`. Use `int()` or `float()` to
  convert them (week 3).

## Putting it together: a script that reads a file

Arguments are very often file names. This script counts the lines of each
file it is given:

In [ ]:
%%writefile count.py
import sys

for filename in sys.argv[1:]:
    n = 0
    with open(filename) as f:
        for line in f:
            n += 1
    print(filename, n)

`sys.argv[1:]` is a slice (week 3): all arguments except the script name.
Run it in the terminal:

```
$ python3 count.py sequences.txt
sequences.txt 3
$ python3 count.py *.txt
```

With the wildcard, the shell gives the script every `.txt` file in the
directory. One command, all files. This is the "doing it 1000 times" from
lesson 1.

## Exit codes

Every program, when it finishes, gives back one whole number: its **exit
code**. `0` means "everything went well". Any other number means an error.
The terminal does not print it, but `echo $?` shows the exit code of the
command you ran just before:

```
$ python3 count.py sequences.txt
sequences.txt 3
$ echo $?
0
```

Now give it a file that does not exist:

```
$ python3 count.py nothere.txt
Traceback (most recent call last):
  ...
FileNotFoundError: [Errno 2] No such file or directory: 'nothere.txt'
$ echo $?
1
```

Python prints the error message and ends with exit code 1. Programs that run
other programs, for example an analysis pipeline, use the exit code to know
whether a step worked. Remember: `echo $?` must come *directly* after the
command you are interested in.

## Summary

- A script is a `.py` text file. Run it with `python3 name.py`.
- A script shows only what it prints.
- `%%writefile name.py` at the top of a cell saves the cell into a file.
- `with open(name, "w") as f:` and `f.write(text)` write a file;
  `with open(name) as f:` and `f.read()` or `for line in f:` read it.
- `sys.argv` is the list of command-line arguments, all strings.
- `echo $?` shows the exit code of the last command: `0` means success.

## Going further (optional)

- In a notebook, a line starting with `!` runs a terminal command, e.g.
  `!ls` or `!python3 count.py sequences.txt`. Handy, but on Windows the
  commands are different.
- A script can end itself with a chosen exit code: `sys.exit(2)`. Use it with
  a helpful message, for example when an argument is missing:
  ```python
  import sys
  if len(sys.argv) < 2:
      print("usage: python3 count.py FILE ...")
      sys.exit(2)
  ```
- For scripts with many options, the module
  [`argparse`](https://docs.python.org/3/library/argparse.html) builds
  `--help` texts and checks the arguments for you.
- Mode `"a"` in `open(name, "a")` **a**dds to the end of a file instead of
  replacing it (like `>>`).

## Check yourself

Answer these without running anything first. Then try them.

1. `add.py` contains
   ```python
   import sys
   print(sys.argv[1] + sys.argv[2])
   ```
   What does `python3 add.py 5 7` print?
2. `python3 hello.py` printed `Hello from a script`. What does `echo $?` print
   directly afterwards?
3. Ben's script `mean.py` contains only
   ```python
   values = [3, 4, 5]
   sum(values) / len(values)
   ```
   He runs `python3 mean.py` and sees nothing. Why? How does he fix it?

Answers are at the end of this notebook.

## Answers to check yourself

1. `57`. Arguments are strings, so `+` joins them. For 12, convert first:
   `int(sys.argv[1]) + int(sys.argv[2])`.
2. `0`, because the script finished without an error.
3. A script does not show the value of the last line; only a notebook does.
   Use `print(sum(values) / len(values))`.